CREATE STREAMING TABLES WITH SQL USING AUTO LOADER INCREMENTALLY


Recommendation: Use it as alternative to the legacy COPY INTO SQL command. DB recommends using streaming tables to ingest data using Databricks SQL

**REQUIRED - SQL COMPUTE

In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/workspace/spotify/databricks_projects/dataset',
  format => 'CSV',
  header => true
);


In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE datasets_inc_str
SCHEDULE EVERY 1 WEEK
AS
SELECT *
FROM STREAM read_files(
  '/Volumes/workspace/spotify/databricks_projects',
  format => 'CSV',
  header => true
)

In [0]:
%sql
DESCRIBE TABLE EXTENDED datasets_inc_str

In [0]:
%sql
DESCRIBE HISTORY datasets_inc_str

In [0]:
%sql
DROP TABLE IF EXISTS datasets_inc_str

Appendix - Python Auto Loader


**Require - Classic compute

In [0]:
spark.sql(f'LIST "/Volumes/workspace/spotify/databricks_projects"').display()

In [0]:
## Create a Volume to store the Auto Loader checkpoint files
spark.sql(f'CREATE VOLUME IF NOT EXISTS workspace.spotify.auto_loader_file')

## Set checkpoint location to the volume above
checkpoint_location = f'/volumes/workspace/spotify/auto_loader_file'

## Incrementally (or stream) data using Auto Loader
(spark
 .readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("cloudFiles.schemaLocation", f"{checkpoint_location}")
    .load(f"/Volumes/workspace/spotify/databricks_projects/")
    .option("checkpointlocation", f"{checkpoint_location}")
    .trigger(once=True)
    .toAble(f"workspace.spotify.python_csv_autoloader")
)

In [0]:
spark.sql(f'DROP VOLUME IF EXISTS workspace.spotify.auto_loader_file')